In [4]:
pip install pandas numpy scikit-learn scipy openpyxl odfpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.0/717.0 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for odfpy: filename=odfpy-1.4.1-py2.py3-none-any.whl size=160673 sha256=f3757a4508ce678257fa6f54a9b40a62073739e561230a613c4df9ce2ec5ae99
  Stored in directory: /root/.cache/pip/wheels/36/5d/63/8243a7ee78fff0f944d638fd0e66d7278888f5e2285d7346b6
Successfully built odfpy


In [2]:
from google.colab import files

print("Upload tes fichiers : ADPs.csv, BCs.csv, etab-ensprimaire-public-men-2.ods")
uploaded = files.upload()

# Vérifier les fichiers uploadés
import os
print("\nFichiers disponibles :")
for f in uploaded.keys():
    print(f"  ✅ {f}")

Upload tes fichiers : ADPs.csv, BCs.csv, etab-ensprimaire-public-men-2.ods


Saving ADP_BP.xlsx to ADP_BP.xlsx
Saving ADPs.csv to ADPs.csv
Saving BCs.csv to BCs.csv
Saving BCs_ADPs_Dp_Agadir_01082025.qgz to BCs_ADPs_Dp_Agadir_01082025.qgz
Saving etab-ensprimaire-public-men-2.ods to etab-ensprimaire-public-men-2.ods

Fichiers disponibles :
  ✅ ADP_BP.xlsx
  ✅ ADPs.csv
  ✅ BCs.csv
  ✅ BCs_ADPs_Dp_Agadir_01082025.qgz
  ✅ etab-ensprimaire-public-men-2.ods


In [5]:
"""
=============================================================================
PROJET: Distribution d'Aide Humanitaire - Agadir, Maroc
Phase 1: Preprocessing des Données
Phase 2: Clustering (K-means, DBSCAN, p-median approx.)
=============================================================================
"""

import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 65)
print("  PROJET AGADIR - DISTRIBUTION AIDE HUMANITAIRE")
print("=" * 65)

# =========================================================================
# PHASE 1 : PREPROCESSING DES DONNÉES
# =========================================================================
print("\n PHASE 1 : PREPROCESSING DES DONNÉES")
print("-" * 65)

# --- T1 : Charger et nettoyer ADPs.csv et BCs.csv ---
print("\n[T1] Chargement et nettoyage ADPs.csv et BCs.csv...")

adps = pd.read_csv("ADPs.csv")
bcs  = pd.read_csv("BCs.csv")

adps.columns = adps.columns.str.strip()
bcs.columns  = bcs.columns.str.strip()

adps = adps.dropna(subset=['Lat', 'Long']).drop_duplicates(subset=['ID']).reset_index(drop=True)
bcs  = bcs.dropna(subset=['Lat', 'Long']).drop_duplicates(subset=['ID']).reset_index(drop=True)

adps = adps.rename(columns={'PLACE_NAME': 'name', 'Lat': 'lat', 'Long': 'lon'})
bcs  = bcs.rename(columns={'Place_Name': 'name', 'Lat': 'lat', 'Long': 'lon'})

print(f"   ADPs : {len(adps)} points")
print(f"   BCs  : {len(bcs)} points")

# --- T2 : Filtrer écoles Agadir depuis ODS ---
print("\n[T2] Filtrage des écoles Agadir depuis le fichier ODS...")

ods_all = pd.read_excel("etab-ensprimaire-public-men-2.ods", engine='odf')
ecoles_agadir = ods_all[ods_all['PROVINCE'] == 'Agadir Ida Outanane'].copy().reset_index(drop=True)
ecoles_agadir = ecoles_agadir.rename(columns={
    'NOM_ETABLISSENTFR': 'nom_fr',
    'NOM_ETABLISSENTAR': 'nom_ar',
    'ADRESSEFR': 'adresse_fr',
    'PROVINCE': 'province',
    'REGIN': 'region'
})
print(f"   {len(ecoles_agadir)} écoles filtrées (Province: Agadir Ida Outanane)")

# --- T3 : Géocodage (proxy bbox Agadir - remplacer par Nominatim en production) ---
print("\n[T3] Géocodage des écoles (proxy bbox Agadir)...")

np.random.seed(42)
n_ecoles = len(ecoles_agadir)
ecoles_agadir['lat'] = np.random.uniform(30.36, 30.47, n_ecoles)
ecoles_agadir['lon'] = np.random.uniform(-9.68, -9.52, n_ecoles)

# En production, décommenter ceci :
# from geopy.geocoders import Nominatim
# from geopy.extra.rate_limiter import RateLimiter
# geolocator = Nominatim(user_agent="agadir_project")
# geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
# ecoles_agadir['location'] = ecoles_agadir['adresse_fr'].apply(
#     lambda x: geocode(f"{x}, Agadir, Maroc"))
# ecoles_agadir['lat'] = ecoles_agadir['location'].apply(
#     lambda x: x.latitude if x else None)
# ecoles_agadir['lon'] = ecoles_agadir['location'].apply(
#     lambda x: x.longitude if x else None)

print(f"   Coordonnées estimées pour {n_ecoles} écoles")

# --- T4 : Estimation de la demande par BC ---
print("\n[T4] Estimation de la demande par BC...")

CENTER_LAT, CENTER_LON = 30.4202, -9.5981

bcs['dist_centre'] = np.sqrt(
    (bcs['lat'] - CENTER_LAT)**2 + (bcs['lon'] - CENTER_LON)**2
)

d_min = bcs['dist_centre'].min()
d_max = bcs['dist_centre'].max()
bcs['demand'] = (
    15000 - (bcs['dist_centre'] - d_min) / (d_max - d_min) * 12000
).round().astype(int)

# Alternative : estimation par type de lieu (nom du BC)
# def estimate_demand_by_name(name):
#     n = str(name).lower()
#     if 'parking' in n: return 150
#     if 'jardin'  in n: return 80
#     if 'place'   in n: return 200
#     if 'mssala'  in n: return 100
#     if 'mosque'  in n: return 120
#     return 90
# bcs['demand'] = bcs['name'].apply(estimate_demand_by_name)

print(f"   Population totale estimée : {bcs['demand'].sum():,} hab")
print(f"   Min/Moy/Max : {bcs['demand'].min()} / {bcs['demand'].mean():.0f} / {bcs['demand'].max()}")

# --- T5 : Matrice de distances Haversine (77 x 27) ---
print("\n[T5] Calcul matrice Haversine (77 BCs × 27 ADPs)...")

def haversine_matrix(lats1, lons1, lats2, lons2):
    R = 6371.0
    lat1 = np.radians(lats1[:, np.newaxis])
    lon1 = np.radians(lons1[:, np.newaxis])
    lat2 = np.radians(lats2[np.newaxis, :])
    lon2 = np.radians(lons2[np.newaxis, :])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

dist_matrix = haversine_matrix(
    bcs['lat'].values, bcs['lon'].values,
    adps['lat'].values, adps['lon'].values
)

print(f"   Matrice : {dist_matrix.shape}")
print(f"   Distance min/moy/max : {dist_matrix.min():.2f} / {dist_matrix.mean():.2f} / {dist_matrix.max():.2f} km")

# --- T6 : Indice de vulnérabilité par ADP ---
print("\n[T6] Calcul indice de vulnérabilité par ADP...")

DEPOT_LAT, DEPOT_LON = 30.3251, -9.4130

adps['dist_depot_km'] = np.sqrt(
    ((adps['lat'] - DEPOT_LAT) * 111)**2 +
    ((adps['lon'] - DEPOT_LON) * 111 * np.cos(np.radians(DEPOT_LAT)))**2
)

# Calcul population à portée (BCs dans rayon 2km)
adp_pop_nearby = np.zeros(len(adps))
for j in range(len(adps)):
    mask = dist_matrix[:, j] < 2.0
    adp_pop_nearby[j] = bcs.loc[mask, 'demand'].sum()

# Centralité
adp_dist_center = np.sqrt(
    ((adps['lat'] - adps['lat'].mean()) * 111)**2 +
    ((adps['lon'] - adps['lon'].mean()) * 111)**2
)

def norm01(x):
    mn, mx = x.min(), x.max()
    return (x - mn) / (mx - mn + 1e-9)

# Formule composite : 50% densité pop + 30% distance dépôt + 20% centralité
adps['vulnerability'] = (
    0.5 * norm01(adp_pop_nearby) +
    0.3 * norm01(adps['dist_depot_km'].values) +
    0.2 * norm01(adp_dist_center)
).round(4)

print(f"   Indice vulnerability : min={adps['vulnerability'].min():.4f} | max={adps['vulnerability'].max():.4f}")
print("   TOP 3 ADPs les plus vulnérables :")
for _, row in adps.nlargest(3, 'vulnerability')[['name','vulnerability','dist_depot_km']].iterrows():
    print(f"      {row['name']} → {row['vulnerability']:.4f} (dist dépôt: {row['dist_depot_km']:.1f}km)")

# --- T7 : Dépôt - Aéroport Al Massira Agadir ---
print("\n[T7] Dépôt : Aéroport Al Massira Agadir")

depot = {
    'name': 'Aeroport_Al_Massira_Agadir',
    'lat':  30.3251,
    'lon':  -9.4130,
    'IATA': 'AGA'
}
print(f"   Coordonnées : {depot['lat']}°N, {depot['lon']}°E  (IATA: {depot['IATA']})")

print("\n✅ PHASE 1 TERMINÉE")

# =========================================================================
# PHASE 2 : CLUSTERING
# =========================================================================
print("\n" + "=" * 65)
print("  PHASE 2 : CLUSTERING")
print("-" * 65)

# Combiner BCs + ADPs pour K-means
all_coords = pd.concat([
    bcs[['lat','lon']].assign(type='BC',  id=bcs['ID'],  name=bcs['name']),
    adps[['lat','lon']].assign(type='ADP', id=adps['ID'], name=adps['name'])
], ignore_index=True)

scaler = StandardScaler()
coords_scaled    = scaler.fit_transform(all_coords[['lat','lon']])
coords_bc_scaled = scaler.transform(bcs[['lat','lon']])

# --- T1 : K-means k=5, 7, 9 ---
print("\n[T1] K-means (k=5, 7, 9) sur BCs+ADPs...")

kmeans_results = {}
for k in [5, 7, 9]:
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
    labels = km.fit_predict(coords_scaled)
    sil = silhouette_score(coords_scaled, labels)
    kmeans_results[k] = {'model': km, 'labels': labels, 'silhouette': sil}
    sizes = pd.Series(labels).value_counts().sort_index().tolist()
    print(f"   k={k} → silhouette={sil:.4f} | tailles: {sizes}")

best_k = max(kmeans_results, key=lambda k: kmeans_results[k]['silhouette'])
print(f"   ★ Meilleur k = {best_k} (silhouette={kmeans_results[best_k]['silhouette']:.4f})")

bcs['kmeans_7']  = kmeans_results[7]['labels'][:len(bcs)]
adps['kmeans_7'] = kmeans_results[7]['labels'][len(bcs):]

# --- T2 : DBSCAN sur BCs ---
print("\n[T2] DBSCAN sur BCs...")

# eps=0.03 ≈ 3km, min_samples=3
db = DBSCAN(eps=0.03, min_samples=3, metric='euclidean')
db_labels = db.fit_predict(coords_bc_scaled)

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()

if n_clusters_db > 1:
    mask_valid = db_labels != -1
    sil_db = silhouette_score(coords_bc_scaled[mask_valid], db_labels[mask_valid])
else:
    sil_db = -1.0

bcs['dbscan'] = db_labels
print(f"   {n_clusters_db} clusters | {n_noise} points bruit (label=-1)")
print(f"   Silhouette : {sil_db:.4f}")
print(f"   Distribution : {dict(pd.Series(db_labels).value_counts().sort_index())}")

# --- T3 : P-median greedy (approx. MILP, p=7) ---
print("\n[T3] P-median greedy (p=7 médians parmi ADPs)...")

def p_median_greedy(dist_mat, demands, p):
    n_bc, n_adp = dist_mat.shape
    selected = []
    # Premier médian : minimise la distance totale pondérée
    cost = np.sum(demands[:, None] * dist_mat, axis=0)
    selected.append(int(np.argmin(cost)))
    # Ajouter successivement les médians qui réduisent le plus le coût
    for _ in range(p - 1):
        current_dist = dist_mat[:, selected].min(axis=1)
        best_gain, best_adp = -np.inf, -1
        for j in range(n_adp):
            if j in selected:
                continue
            new_dist = np.minimum(current_dist, dist_mat[:, j])
            gain = np.sum(demands * (current_dist - new_dist))
            if gain > best_gain:
                best_gain, best_adp = gain, j
        selected.append(best_adp)
    selected = sorted(selected)
    # Assignation : chaque BC → ADP médian le plus proche
    sub_dist = dist_mat[:, selected]
    assignments = np.argmin(sub_dist, axis=1)
    mapping = {adp: i for i, adp in enumerate(selected)}
    labels = np.array([mapping[selected[a]] for a in assignments])
    return selected, labels

p_medians, pmed_labels = p_median_greedy(dist_matrix, bcs['demand'].values, p=7)
bcs['pmedian_7'] = pmed_labels

sil_pmed = silhouette_score(coords_bc_scaled, pmed_labels) if len(set(pmed_labels)) > 1 else -1.0
cost_pmed = sum(
    bcs['demand'].iloc[i] * dist_matrix[i, p_medians[pmed_labels[i]]]
    for i in range(len(bcs))
)

print(f"   ADPs médians sélectionnés : {[adps.iloc[j]['name'] for j in p_medians]}")
print(f"   Coût total pondéré : {cost_pmed:.1f} km·hab")
print(f"   Silhouette : {sil_pmed:.4f}")
print(f"   Tailles clusters : {pd.Series(pmed_labels).value_counts().sort_index().tolist()}")

# --- T4 : Tableau comparatif ---
print("\n[T4] Tableau comparatif des méthodes...")

comparison = pd.DataFrame({
    'Methode':     ['K-means k=5', 'K-means k=7', 'K-means k=9', 'DBSCAN', 'P-median p=7'],
    'N_clusters':  [5, 7, 9, n_clusters_db, 7],
    'Silhouette':  [round(kmeans_results[k]['silhouette'], 4) for k in [5,7,9]] + [round(sil_db,4), round(sil_pmed,4)],
    'Points_bruit':[0, 0, 0, n_noise, 0]
})
print(comparison.to_string(index=False))
best_method = comparison.loc[comparison['Silhouette'].idxmax(), 'Methode']
print(f"\n   ★ Meilleure méthode : {best_method}")

# --- T5 : Export GeoJSON pour QGIS ---
print("\n[T5] Export GeoJSON pour QGIS...")

def to_geojson(df, lat_col, lon_col, props_cols, filepath):
    features = []
    for _, row in df.iterrows():
        props = {}
        for col in props_cols:
            v = row[col]
            if isinstance(v, (np.integer,)): props[col] = int(v)
            elif isinstance(v, (np.floating,)): props[col] = float(v)
            else: props[col] = str(v)
        features.append({
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [float(row[lon_col]), float(row[lat_col])]},
            "properties": props
        })
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump({"type": "FeatureCollection", "features": features}, f, ensure_ascii=False, indent=2)
    print(f"   ✅ {os.path.basename(filepath)} ({len(features)} features)")

to_geojson(bcs,  'lat', 'lon', ['ID','name','demand','kmeans_7','dbscan','pmedian_7'], f"{OUTPUT_DIR}/BCs_clusters.geojson")
to_geojson(adps, 'lat', 'lon', ['ID','name','City','vulnerability','kmeans_7'],        f"{OUTPUT_DIR}/ADPs_clusters.geojson")
to_geojson(ecoles_agadir, 'lat', 'lon', ['nom_fr','adresse_fr','province'],             f"{OUTPUT_DIR}/Ecoles_Agadir_105.geojson")

# Dépôt
import json as _json
with open(f"{OUTPUT_DIR}/Depot_Agadir.geojson", 'w') as f:
    _json.dump({"type":"FeatureCollection","features":[{
        "type":"Feature",
        "geometry":{"type":"Point","coordinates":[depot['lon'], depot['lat']]},
        "properties": depot
    }]}, f, indent=2)
print(f"   ✅ Depot_Agadir.geojson")

# --- T6 : Rapport équilibre des clusters ---
print("\n[T6] Validation équilibre des clusters (K-means k=7)...")
print(f"\n  {'Cluster':>8} | {'BCs':>4} | {'ADPs':>4} | {'Demande':>10} | {'Vuln moy':>9}")
print("  " + "-"*45)
for c in range(7):
    bc_m  = bcs['kmeans_7'] == c
    adp_m = adps['kmeans_7'] == c
    dem   = bcs.loc[bc_m, 'demand'].sum()
    vuln  = adps.loc[adp_m, 'vulnerability'].mean() if adp_m.sum() > 0 else 0
    print(f"  {c:>8} | {bc_m.sum():>4} | {adp_m.sum():>4} | {dem:>10,} | {vuln:>9.3f}")

# --- Sauvegarde CSV ---
print("\n[SAVE] Sauvegarde CSV...")
bcs.to_csv(f"{OUTPUT_DIR}/BCs_clean.csv", index=False)
adps.to_csv(f"{OUTPUT_DIR}/ADPs_clean.csv", index=False)
ecoles_agadir.to_csv(f"{OUTPUT_DIR}/Ecoles_Agadir_105.csv", index=False)
comparison.to_csv(f"{OUTPUT_DIR}/Rapport_Clustering.csv", index=False)
print(f"   ✅ BCs_clean.csv | ADPs_clean.csv | Ecoles_Agadir_105.csv | Rapport_Clustering.csv")

# =========================================================================
# RÉSUMÉ FINAL
# =========================================================================
print("\n" + "=" * 65)
print("  RÉSUMÉ FINAL")
print("=" * 65)
print(f"""
  PHASE 1:
  ✅ T1 : {len(bcs)} BCs + {len(adps)} ADPs nettoyés
  ✅ T2 : {len(ecoles_agadir)} écoles Agadir filtrées
  ✅ T3 : Géocodage écoles (proxy — Nominatim en production)
  ✅ T4 : demand → total {bcs['demand'].sum():,} habitants
  ✅ T5 : Matrice Haversine {dist_matrix.shape[0]}×{dist_matrix.shape[1]}
  ✅ T6 : vulnerability par ADP (0–1)
  ✅ T7 : Dépôt = Aéroport Al Massira ({depot['lat']}°N, {depot['lon']}°E)

  PHASE 2:
  ✅ T1 : K-means k=5,7,9 → meilleur k={best_k}
  ✅ T2 : DBSCAN → {n_clusters_db} clusters, {n_noise} bruit
  ✅ T3 : P-median greedy p=7
  ✅ T4 : Meilleure méthode → {best_method}
  ✅ T5 : GeoJSON exportés pour QGIS
  ✅ T6 : Rapport équilibre clusters

  FICHIERS → dossier '{OUTPUT_DIR}/' :
  BCs_clean.csv | ADPs_clean.csv | Ecoles_Agadir_105.csv
  Rapport_Clustering.csv
  BCs_clusters.geojson | ADPs_clusters.geojson
  Depot_Agadir.geojson | Ecoles_Agadir_105.geojson
""")

  PROJET AGADIR - DISTRIBUTION AIDE HUMANITAIRE

 PHASE 1 : PREPROCESSING DES DONNÉES
-----------------------------------------------------------------

[T1] Chargement et nettoyage ADPs.csv et BCs.csv...
   ADPs : 27 points
   BCs  : 77 points

[T2] Filtrage des écoles Agadir depuis le fichier ODS...
   105 écoles filtrées (Province: Agadir Ida Outanane)

[T3] Géocodage des écoles (proxy bbox Agadir)...
   Coordonnées estimées pour 105 écoles

[T4] Estimation de la demande par BC...
   Population totale estimée : 907,953 hab
   Min/Moy/Max : 3000 / 11792 / 15000

[T5] Calcul matrice Haversine (77 BCs × 27 ADPs)...
   Matrice : (77, 27)
   Distance min/moy/max : 0.07 / 7.85 / 20.69 km

[T6] Calcul indice de vulnérabilité par ADP...
   Indice vulnerability : min=0.0968 | max=0.7580
   TOP 3 ADPs les plus vulnérables :
      L_LALLA_MERIYEM → 0.7580 (dist dépôt: 21.4km)
      L_AL_MOUTANABI → 0.7488 (dist dépôt: 20.3km)
      L_ABDELKARIM__AL_KHATTABI → 0.5550 (dist dépôt: 18.2km)

[T7] 